[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# groupby &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import math
import re
import warnings
from datetime import datetime, timedelta

import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})


def print_warnings(caught):
    """Print each caught warning as its category and the first sentence of its message."""
    for warning in caught:
        first_line = str(warning.message).strip().splitlines()[0]
        print(f"{warning.category.__name__}: {first_line.split('. ')[0].rstrip('.')}.")


print("pandas", pd.__version__, "|", len(readings), "readings,", len(stations), "stations")

pandas 2.2.3 | 35040 readings, 5 stations


**1.** Print each station's lowest reading of the year, with `groupby` and `min`.

In [2]:
readings.groupby("station")["celsius"].min()

station
Bergen      -4.8
Oslo        -6.3
Svalbard   -17.3
Tromso      -9.2
Name: celsius, dtype: float64

`min` leaves the missing readings out, so Svalbard's lowest is a real reading.

**2.** Count the readings below zero at each station, by filtering first and grouping the result.

In [3]:
below_zero = readings[readings["celsius"] < 0]

below_zero.groupby("station").size()

station
Bergen      1220
Oslo        1848
Svalbard    5871
Tromso      3203
dtype: int64

The filter keeps the rows below zero, and `size` counts what is left in each group. Svalbard's missing hours are not below zero, so they are not counted.

**3.** Print the mean of each month across all four stations, rounded to one place, by grouping on `readings["time"].dt.month`.

In [4]:
readings.groupby(readings["time"].dt.month)["celsius"].mean().round(1)

time
1     -5.5
2     -4.5
3     -1.4
4      2.9
5      7.4
6     10.9
7     12.3
8     11.2
9      8.1
10     3.7
11    -0.8
12    -4.2
Name: celsius, dtype: float64

The key is a Series of months, one for every reading, so every station's readings for a month fall in the same group.

**4.** Loop over `readings.groupby("station")` and print each station's name with the number of its readings above 10 degrees.

In [5]:
for station, group in readings.groupby("station"):
    print(station, (group["celsius"] > 10).sum())

Bergen 3718
Oslo 3197
Svalbard 0
Tromso 1847


`group["celsius"] > 10` is a mask, and `sum` counts its `True` values.

**5.** Make a table, with `as_index=False`, of each station's highest reading, sorted from highest to lowest.

In [6]:
highest = readings.groupby("station", as_index=False)["celsius"].max()

highest.sort_values("celsius", ascending=False)

,station,celsius
0,Bergen,20.8
1,Oslo,19.3
3,Tromso,16.2
2,Svalbard,8.3


With `as_index=False` the stations are a column, and `sort_values` orders the rows by the highest reading. No two stations share a highest reading, so the order is the same on every run.

---

&#8592; **Back to:** [groupby](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/12-groupby.ipynb)  &nbsp;&middot;&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)
